In [5]:
import os, re, math, numpy as np, pandas as pd, warnings
warnings.filterwarnings("ignore")

# =======================
# CONFIG
# =======================
DATA_PATH = "/root/test/score/raw_data/user_data.csv"
OUT_DIR   = "/root/test/score/output"

# Tạo thư mục output nếu chưa tồn tại
os.makedirs(OUT_DIR, exist_ok=True)

TARGET_CANDIDATES = ["BAD", "BAD_lock"]   # data mới bạn nói có BAD, nếu không có thì dùng BAD_lock
DATE_COL = "loan_date"                    # dùng để time-split (nếu có)

# BỎ TỈNH theo yêu cầu
DROP_ALWAYS = {"vnpostProvinceName", "vnpostUsername", "vnpostUsername_ne", "reasonLock"}

# Chỉ xét các feature “hành vi/velocity/trigger” để SQL gọn + chạy nhanh
KEEP_PATTERNS = [
    r"^app_cnt_\d+d$",
    r"^approve_cnt_\d+d$",
    r"^outSideApp_cnt_\d+d$",
    r"^outSideAppPermanent_cnt_\d+d$",
    r"^term_permanent_flag_cnt_\d+d$",
    r"^idTrigger_cnt_\d+d$",
    r"^phoneTrigger_cnt_\d+d$",
    r"^userTriggerNew_cnt_\d+d$",
    r"^createTimeTrigger_cnt_\d+d$",
    r"^addressTrigger_cnt_\d+d$",
    r"^idcard_cnt_\d+d$",
    r"^idcard_rej_cnt_\d+d$",
    r"^male_cnt_\d+d$",
    r"^Single_cnt_\d+d$",
    r"^amt30_cnt_\d+d$",
    r"^refphone_cnt_\d+d$",
    r"^nonViettel_cnt_\d+d$",
    r"^acceleration_\d+d$",
    r"^velocity_\d+d$",
    r"^avg_value_ratio$",
    r"^metrics_exceeded$",
    r"^days_since_last_activity$",
    r"^region_tier$",
]

# Drop mấy cột kiểu scale/constant
DROP_REGEX = [r".*_total$", r".*_pct$", r"^pct_.*"]

MISSING_DROP_TH = 0.995

# Binning
NUM_BINS_MAX = 6
MIN_BIN_PCT  = 0.03
CAT_MIN_COUNT_PCT = 0.01

# Feature selection
IV_MIN_KEEP = 0.02
REGION_TIER_IV_KEEP = 0.05   # "thực sự cao": bạn có thể nâng lên 0.1 nếu muốn chặt hơn
MAX_FEATURES_AFTER_IV = 20
CORR_DROP_TH = 0.85

# Logistic
from sklearn.linear_model import LogisticRegression
LR_C_SELECT = 0.5   # L1 chọn biến
LR_C_FINAL  = 1.0   # L2 fit cuối

# Score scaling
BASE_SCORE = 600
PDO = 50
BASE_ODDS = None    # None => dùng good/bad của TRAIN

# =======================
# HELPERS
# =======================
def pick_target(df: pd.DataFrame) -> str:
    for c in TARGET_CANDIDATES:
        if c in df.columns:
            return c
    raise ValueError(f"Không tìm thấy target trong {TARGET_CANDIDATES}")

def matches_any(name: str, patterns):
    return any(re.match(p, name) for p in patterns)

def drop_by_regex(cols, patterns):
    out=set()
    for c in cols:
        for rx in patterns:
            if re.match(rx,c):
                out.add(c); break
    return out

def safe_to_numeric(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s, errors="coerce")

def numeric_ok(s: pd.Series, th=0.7):
    x = safe_to_numeric(s)
    return x.notna().mean() >= th

def woe_iv_from_bin_ids(bin_id: np.ndarray, y: np.ndarray, eps=1e-6):
    y = y.astype(int)
    bins = np.unique(bin_id)
    total_bad  = (y==1).sum()
    total_good = (y==0).sum()
    m = len(bins)

    woe_map={}
    rows=[]
    iv=0.0
    for b in bins:
        mask = (bin_id==b)
        bad  = int((y[mask]==1).sum())
        good = int((y[mask]==0).sum())
        cnt  = int(mask.sum())

        dist_bad  = (bad  + eps) / (total_bad  + eps*m)
        dist_good = (good + eps) / (total_good + eps*m)
        woe = math.log(dist_good/dist_bad)
        iv_bin = (dist_good - dist_bad) * woe
        iv += iv_bin

        woe_map[int(b)] = woe
        rows.append((int(b), cnt, bad, good, (bad/cnt if cnt else np.nan), woe, iv_bin))

    stats = pd.DataFrame(rows, columns=["bin_id","count","bad","good","bad_rate","woe","iv_bin"]).sort_values("bin_id")
    return woe_map, iv, stats

def fit_numeric_bins_np(x: np.ndarray, y: np.ndarray, max_bins=6, min_bin_pct=0.03):
    n = len(x)
    miss = ~np.isfinite(x)
    x_nm = x[~miss]
    if x_nm.size < 50:
        return np.where(miss,-1,0).astype(int), np.array([],dtype=float)

    bins = max_bins
    while bins >= 2:
        qs = np.linspace(0,1,bins+1)
        cuts = np.unique(np.quantile(x_nm, qs)[1:-1])  # inner cutpoints
        if cuts.size == 0:
            bins -= 1; continue

        b = np.digitize(x, cuts, right=True).astype(int)  # 0..k
        b = np.where(miss, -1, b).astype(int)

        b_nm = b[b!=-1]
        if b_nm.size == 0:
            bins -= 1; continue
        counts = np.bincount(b_nm)
        if counts.min() < int(min_bin_pct*n):
            bins -= 1; continue

        return b, cuts

    return np.where(miss,-1,0).astype(int), np.array([],dtype=float)

def bin_labels_from_cutpoints(cuts: np.ndarray):
    if cuts.size==0:
        return ["__ALL__"]
    labels=[]
    prev="-inf"
    for c in cuts:
        labels.append(f"({prev}, {c}]")
        prev=str(c)
    labels.append(f"({prev}, inf)")
    return labels

def apply_numeric_bins_np(x: np.ndarray, cuts: np.ndarray):
    miss = ~np.isfinite(x)
    if cuts.size==0:
        return np.where(miss,-1,0).astype(int)
    b = np.digitize(x, cuts, right=True).astype(int)
    return np.where(miss,-1,b).astype(int)

def fit_cat_bins(s: pd.Series, min_count_pct=0.01):
    s = s.astype("string").fillna("__MISSING__")
    n = len(s)
    vc = s.value_counts()
    keep = set(vc[vc >= int(min_count_pct*n)].index)
    keep.add("__MISSING__")
    return s.where(s.isin(keep), "__OTHER__")

def encode_cat_to_ids(s2: pd.Series):
    cats = pd.unique(s2)
    mapping = {c:i for i,c in enumerate(cats)}
    ids = s2.map(mapping).to_numpy(dtype=int)
    return ids, mapping

def corr_prune(df_woe: pd.DataFrame, ivs: pd.Series, th=0.85):
    feats = list(ivs.sort_values(ascending=False).index)
    corr = df_woe[feats].corr().abs()
    keep=[]
    for f in feats:
        ok = True
        for k in keep:
            if pd.notna(corr.loc[f,k]) and corr.loc[f,k] >= th:
                ok=False; break
        if ok:
            keep.append(f)
    return keep

def sql_escape(s: str) -> str:
    return s.replace("\\","\\\\").replace("'","\\'")

# =======================
# LOAD
# =======================
assert os.path.exists(DATA_PATH), f"Không thấy file: {DATA_PATH}"
df = pd.read_csv(DATA_PATH, low_memory=False)

target = pick_target(df)
y = df[target].astype(int).to_numpy()

# =======================
# TIME SPLIT (80/20 theo loan_date nếu có)
# =======================
cutoff_date = None
if DATE_COL in df.columns:
    d = pd.to_datetime(df[DATE_COL], errors="coerce")
    if d.notna().mean() > 0.1:
        order = d.sort_values().index.to_numpy()
        cut_idx = int(0.8 * len(order))
        cutoff_date = d.iloc[order[cut_idx]]
        train_mask = ((d <= cutoff_date) | d.isna()).to_numpy()
    else:
        rng = np.random.RandomState(42)
        train_mask = (rng.rand(len(df)) < 0.8)
else:
    rng = np.random.RandomState(42)
    train_mask = (rng.rand(len(df)) < 0.8)

test_mask = ~train_mask

# =======================
# CANDIDATES
# =======================
cols = df.columns.tolist()
cand = [c for c in cols if matches_any(c, KEEP_PATTERNS)]
drop = set(DROP_ALWAYS) | drop_by_regex(cols, DROP_REGEX) | {target}
cand = [c for c in cand if c in df.columns and c not in drop]

# drop high-missing & constant
miss_rate = df[cand].isna().mean()
cand = [c for c in cand if miss_rate[c] <= MISSING_DROP_TH]
nuniq = df[cand].nunique(dropna=False)
cand = [c for c in cand if nuniq[c] > 1]

# =======================
# IV SCREEN (sample train để nhanh)
# =======================
train_idx = np.where(train_mask)[0]
sample_n = min(60_000, len(train_idx))
rng = np.random.RandomState(123)
sample_idx = rng.choice(train_idx, size=sample_n, replace=False)

y_s = y[sample_idx]
ivs={}
ftype={}
region_iv=None

for c in cand:
    s = df.loc[sample_idx, c]
    try:
        if numeric_ok(s):
            x = safe_to_numeric(s).to_numpy(dtype=float)
            b, cuts = fit_numeric_bins_np(x, y_s, max_bins=NUM_BINS_MAX, min_bin_pct=MIN_BIN_PCT)
            _, iv, _ = woe_iv_from_bin_ids(b, y_s)
            ivs[c]=iv; ftype[c]="numeric"
        else:
            s2 = fit_cat_bins(s, min_count_pct=CAT_MIN_COUNT_PCT)
            ids, _ = encode_cat_to_ids(s2)
            _, iv, _ = woe_iv_from_bin_ids(ids, y_s)
            ivs[c]=iv; ftype[c]="categorical"
    except Exception:
        continue

iv_series = pd.Series(ivs).sort_values(ascending=False)
if "region_tier" in iv_series.index:
    region_iv = float(iv_series["region_tier"])

keep = iv_series[iv_series >= IV_MIN_KEEP].index.tolist()
# region_tier chỉ giữ nếu IV đủ cao
if "region_tier" in keep and region_iv is not None and region_iv < REGION_TIER_IV_KEEP:
    keep = [f for f in keep if f != "region_tier"]

keep = keep[:MAX_FEATURES_AFTER_IV]

# =======================
# FIT WOE ON FULL TRAIN (cho các biến đã qua IV)
# =======================
meta={}
woe_df = pd.DataFrame(index=df.index)
bin_rows=[]
y_train = y[train_mask]

for c in keep:
    s_tr = df.loc[train_mask, c]
    if ftype.get(c) == "numeric":
        x_tr = safe_to_numeric(s_tr).to_numpy(dtype=float)
        b_tr, cuts = fit_numeric_bins_np(x_tr, y_train, max_bins=NUM_BINS_MAX, min_bin_pct=MIN_BIN_PCT)
        wmap, iv, stats = woe_iv_from_bin_ids(b_tr, y_train)

        labels = bin_labels_from_cutpoints(cuts)
        def id_to_label(bid):
            if bid==-1: return "__MISSING__"
            if cuts.size==0: return "__ALL__"
            return labels[bid]
        stats["bin_label"] = stats["bin_id"].map(id_to_label)

        x_full = safe_to_numeric(df[c]).to_numpy(dtype=float)
        b_full = apply_numeric_bins_np(x_full, cuts)
        woe_full = np.vectorize(lambda bid: wmap[int(bid)])(b_full).astype(float)
        woe_df[c] = woe_full

        for _, r in stats.iterrows():
            bin_rows.append({
                "feature": c, "type":"numeric",
                "bin_id": int(r["bin_id"]),
                "bin_label": str(r["bin_label"]),
                "woe": float(r["woe"]),
                "iv_bin": float(r["iv_bin"]),
                "count_train": int(r["count"]),
                "bad_train": int(r["bad"]),
                "good_train": int(r["good"]),
                "bad_rate_train": float(r["bad_rate"]) if pd.notna(r["bad_rate"]) else np.nan,
            })

        meta[c] = {"type":"numeric", "cuts":cuts, "labels":labels, "woe_map":wmap, "iv":float(iv)}

    else:
        s_tr2 = fit_cat_bins(s_tr, min_count_pct=CAT_MIN_COUNT_PCT)
        ids_tr, map_tr = encode_cat_to_ids(s_tr2)
        wmap, iv, stats = woe_iv_from_bin_ids(ids_tr, y_train)

        inv_map = {v:k for k,v in map_tr.items()}
        stats["bin_label"] = stats["bin_id"].map(lambda i: str(inv_map.get(int(i), "__OTHER__")))

        s_full = df[c].astype("string").fillna("__MISSING__")
        keep_cats = set(map_tr.keys())
        s_full2 = s_full.where(s_full.isin(keep_cats), "__OTHER__")
        ids_full = s_full2.map(lambda v: map_tr.get(v, map_tr.get("__OTHER__",0))).to_numpy(dtype=int)
        woe_full = np.vectorize(lambda bid: wmap[int(bid)])(ids_full).astype(float)
        woe_df[c] = woe_full

        for _, r in stats.iterrows():
            bin_rows.append({
                "feature": c, "type":"categorical",
                "bin_id": int(r["bin_id"]),
                "bin_label": str(r["bin_label"]),
                "woe": float(r["woe"]),
                "iv_bin": float(r["iv_bin"]),
                "count_train": int(r["count"]),
                "bad_train": int(r["bad"]),
                "good_train": int(r["good"]),
                "bad_rate_train": float(r["bad_rate"]) if pd.notna(r["bad_rate"]) else np.nan,
            })

        meta[c] = {"type":"categorical", "cat_map":map_tr, "woe_map":wmap, "iv":float(iv)}

# =======================
# PRUNE CORR + LOGISTIC (L1 select, L2 final)
# =======================
iv_keep = pd.Series({c: meta[c]["iv"] for c in keep}).sort_values(ascending=False)
keep2 = corr_prune(woe_df.loc[train_mask, keep], iv_keep, th=CORR_DROP_TH)

X_train = woe_df.loc[train_mask, keep2]
X_test  = woe_df.loc[test_mask,  keep2]
y_train = y[train_mask]
y_test  = y[test_mask]

lr_sel = LogisticRegression(penalty="l1", solver="liblinear", C=LR_C_SELECT,
                            class_weight="balanced", max_iter=300)
lr_sel.fit(X_train, y_train)
sel_feats = [f for f,b in zip(keep2, lr_sel.coef_[0]) if abs(b) > 1e-8]
if not sel_feats:
    sel_feats = keep2[:10]

lr = LogisticRegression(penalty="l2", solver="lbfgs", C=LR_C_FINAL,
                        class_weight="balanced", max_iter=300)
lr.fit(woe_df.loc[train_mask, sel_feats], y_train)

from sklearn.metrics import roc_auc_score
p_tr = lr.predict_proba(woe_df.loc[train_mask, sel_feats])[:,1]
p_te = lr.predict_proba(woe_df.loc[test_mask,  sel_feats])[:,1]
auc_tr = roc_auc_score(y_train, p_tr) if len(np.unique(y_train))>1 else np.nan
auc_te = roc_auc_score(y_test,  p_te) if len(np.unique(y_test))>1 else np.nan

# =======================
# SCORE SCALING
# =======================
factor = PDO / np.log(2)
if BASE_ODDS is None:
    bad  = (y_train==1).sum()
    good = (y_train==0).sum()
    base_odds = (good / bad) if bad>0 else 50.0
else:
    base_odds = float(BASE_ODDS)

offset = BASE_SCORE - factor*np.log(base_odds)
intercept = float(lr.intercept_[0])
coefs = {f: float(b) for f,b in zip(sel_feats, lr.coef_[0])}
base_points_int = int(np.round(offset - factor*intercept))

# =======================
# BUILD BINS + POINTS
# =======================
bins_df = pd.DataFrame(bin_rows)
bins_df = bins_df[bins_df["feature"].isin(sel_feats)].copy()
bins_df["beta"] = bins_df["feature"].map(coefs).astype(float)
bins_df["points_raw"] = -factor * bins_df["beta"] * bins_df["woe"]
bins_df["points"] = np.round(bins_df["points_raw"]).astype(int)

feat_summary_df = pd.DataFrame([{
    "feature": f,
    "type": meta[f]["type"],
    "iv_train": meta[f]["iv"],
    "beta": coefs[f],
    "n_bins": int((bins_df["feature"]==f).sum())
} for f in sel_feats]).sort_values("iv_train", ascending=False)

def sql_escape(s: str) -> str:
    return s.replace("\\", "\\\\").replace("'", "\\'")

def build_case_numeric(col, cuts, points_by_label):
    """
    BigQuery CASE cho numeric:
    - dùng SAFE_CAST về FLOAT64
    - missing => points của __MISSING__
    - bins theo cutpoints: <=cut1, <=cut2, ..., ELSE (bin cuối)
    """
    miss_pts = int(points_by_label.get("__MISSING__", 0))
    x = f"SAFE_CAST({col} AS FLOAT64)"

    # labels phải khớp với bin_labels_from_cutpoints(cuts)
    labels = bin_labels_from_cutpoints(cuts)

    lines = ["CASE", f"  WHEN {x} IS NULL THEN {miss_pts}"]

    if cuts.size == 0:
        pts = int(points_by_label.get("__ALL__", 0))
        lines.append(f"  ELSE {pts}")
        lines.append("END")
        return "\n".join(lines)

    for i, c in enumerate(cuts):
        lbl = labels[i]
        pts = int(points_by_label.get(lbl, 0))
        lines.append(f"  WHEN {x} <= {c:.10g} THEN {pts}")

    last_lbl = labels[-1]
    last_pts = int(points_by_label.get(last_lbl, 0))
    lines.append(f"  ELSE {last_pts}")
    lines.append("END")
    return "\n".join(lines)

def build_case_categorical(col, cat_map, points_by_label):
    """
    BigQuery CASE cho categorical:
    - missing => __MISSING__
    - không match => __OTHER__
    """
    miss_pts = int(points_by_label.get("__MISSING__", 0))
    other_pts = int(points_by_label.get("__OTHER__", 0))
    x = f"CAST({col} AS STRING)"

    lines = ["CASE", f"  WHEN {col} IS NULL THEN {miss_pts}"]

    cats = [c for c in cat_map.keys() if c not in ["__MISSING__", "__OTHER__"]]
    for cat in sorted(cats):
        pts = int(points_by_label.get(str(cat), other_pts))
        lit = f"'{sql_escape(str(cat))}'"
        lines.append(f"  WHEN {x} = {lit} THEN {pts}")

    lines.append(f"  ELSE {other_pts}")
    lines.append("END")
    return "\n".join(lines)

# =======================
# SQL GENERATION (BigQuery)
# =======================
points_maps={}
for f in sel_feats:
    sub = bins_df[bins_df["feature"]==f][["bin_label","points"]]
    points_maps[f] = {r["bin_label"]: int(r["points"]) for _,r in sub.iterrows()}

sql=[]
sql.append("-- BigQuery Standard SQL")
sql.append(f"-- Target used: {target}")
if cutoff_date is not None:
    sql.append(f"-- Time-split cutoff_date (train<=): {str(cutoff_date)}")
sql.append(f"-- AUC train: {auc_tr:.6f} | AUC test: {auc_te:.6f}")
sql.append(f"-- base_score={BASE_SCORE}, PDO={PDO}, base_odds={base_odds:.6f}")
sql.append(f"-- base_points={base_points_int}")
sql.append("")
sql.append("WITH scored AS (")
sql.append("  SELECT")
sql.append("    t.*,")
sql.append(f"    {base_points_int} AS base_points,")

for f in sel_feats:
    col = f"`{f}`" if re.search(r"\W", f) else f  # ✅ FIX HERE
    alias = f"pts_{f}"
    if meta[f]["type"]=="numeric":
        case = build_case_numeric(col, meta[f]["cuts"], points_maps[f])
    else:
        case = build_case_categorical(col, meta[f]["cat_map"], points_maps[f])
    sql.append(f"    ({case}) AS {alias},")

sum_expr = " + ".join([f"pts_{f}" for f in sel_feats])
sql.append(f"    ({base_points_int} + {sum_expr}) AS user_score")
sql.append("  FROM `your_table` t")
sql.append(")")
sql.append("SELECT * FROM scored;")
sql_text = "\n".join(sql)


# =======================
# SAVE
# =======================
bins_out = os.path.join(OUT_DIR, "scorecard_bins_no_province.csv")
feat_out = os.path.join(OUT_DIR, "scorecard_feature_summary_no_province.csv")
sql_out  = os.path.join(OUT_DIR, "user_score_scorecard_no_province.sql")

bins_df.sort_values(["feature","bin_id"], inplace=True)
bins_df.to_csv(bins_out, index=False, encoding="utf-8-sig")
feat_summary_df.to_csv(feat_out, index=False, encoding="utf-8-sig")
with open(sql_out, "w", encoding="utf-8") as f:
    f.write(sql_text)

print("✅ DONE")
print("Target:", target)
print("region_tier IV (sample):", region_iv)
print("Giữ region_tier?", ("region_tier" in sel_feats))
print("Final feats:", len(sel_feats), sel_feats)
print("AUC train:", auc_tr, "| AUC test:", auc_te)
print("Saved:")
print(" -", bins_out)
print(" -", feat_out)
print(" -", sql_out)


✅ DONE
Target: BAD_lock
region_tier IV (sample): 0.65611696873971
Giữ region_tier? False
Final feats: 9 ['male_cnt_28d', 'amt30_cnt_28d', 'term_permanent_flag_cnt_28d', 'idcard_cnt_14d', 'velocity_7d', 'nonViettel_cnt_28d', 'velocity_28d', 'avg_value_ratio', 'term_permanent_flag_cnt_7d']
AUC train: 0.7844645892293203 | AUC test: 0.7392770263076996
Saved:
 - /root/test/score/output/scorecard_bins_no_province.csv
 - /root/test/score/output/scorecard_feature_summary_no_province.csv
 - /root/test/score/output/user_score_scorecard_no_province.sql
